# 08 — Knowledge Distillation Experiment

Train a small **student** (Gemma-3-270m) to mimic a larger **teacher**
(Gemma-3-1b-it) using:

- **Hard targets**: the ground-truth next token (cross-entropy loss)
- **Soft targets**: the teacher's full probability distribution (KL divergence)

The student learns the teacher's *dark knowledge* — the relative
probabilities of incorrect tokens — resulting in a much stronger model
than plain fine-tuning.

In [ ]:
import sys
sys.path.append("..")

import torch
import matplotlib.pyplot as plt
from transformers import AutoTokenizer

from src.llm_optimization.core import load_config
from src.llm_optimization.data import load_and_prepare_data, QADataset
from src.llm_optimization.training import build_distillation_trainer
from src.llm_optimization.utils import ResourceMonitor, gpu_mem, peak_gpu_mem, reset_peak_gpu_mem

print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('BF16 supported:', torch.cuda.is_bf16_supported())

In [ ]:
config = load_config('./configs/distillation.yaml')

print('Method:          ', config.method.value)
print('Student model:   ', config.model_name)
print('Teacher model:   ', config.distillation.teacher_model_name)
print('Temperature:     ', config.distillation.temperature)
print('Alpha (distill): ', config.distillation.alpha)
print('Epochs:          ', config.training.num_train_epochs)

In [ ]:
train_df, _, val_df = load_and_prepare_data(config.data)

tokenizer = AutoTokenizer.from_pretrained(config.model_name)
tokenizer.pad_token = tokenizer.eos_token

train_ds = QADataset(train_df, tokenizer, config.data.max_length)
val_ds = QADataset(val_df, tokenizer, config.data.max_length)

print(f'Train: {len(train_ds)}  Val: {len(val_ds)}')
print(f'Vocab size: {len(tokenizer):,}')

## Build the distillation trainer

This loads **both** the teacher (frozen) and the student. Expect ~2× VRAM of the
plain fine-tuning setup.

| Model | Role | Params |
|-------|------|--------|
| gemma-3-1b-it | Teacher (frozen) | ~1B |
| gemma-3-270m  | Student (trainable) | ~270M |

In [ ]:
reset_peak_gpu_mem()

trainer, student = build_distillation_trainer(
    config, train_ds, val_ds, tokenizer
)

total_student = sum(p.numel() for p in student.parameters())
trainable = sum(p.numel() for p in student.parameters() if p.requires_grad)

print(f'\nStudent total params:     {total_student:,}')
print(f'Student trainable params: {trainable:,} ({100*trainable/total_student:.2f}%)')
print(f'Peak VRAM after loading:  {peak_gpu_mem()}')

## Train

The custom `DistillationTrainer` overrides `compute_loss` to compute:

    loss = (1 - α) · CE(student, labels)  +  α · KL(student || teacher)

where α = 0.5 and the KL term uses temperature T = 2.0 to soften distributions.

In [ ]:
monitor = ResourceMonitor()
if torch.cuda.is_available():
    torch.cuda.reset_peak_memory_stats()

result = trainer.train()

monitor.print_summary()
print(f'\nFinal train loss: {result.training_loss:.4f}')
if torch.cuda.is_available():
    print(f'Peak VRAM: {torch.cuda.max_memory_allocated()/1024**2:.0f} MB')

## Evaluate

In [ ]:
eval_metrics = trainer.evaluate()
print('Validation metrics:')
for k, v in eval_metrics.items():
    print(f'  {k}: {v:.4f}')

## Save the student

In [ ]:
trainer.save_model()
tokenizer.save_pretrained(config.output_path)

import os
saved = os.listdir(config.output_path)
print(f'Saved {len(saved)} files to {config.output_path}')
for f in sorted(saved):
    fp = os.path.join(config.output_path, f)
    if os.path.isfile(fp):
        print(f'  {f:<40} {os.path.getsize(fp)/1024**2:>8.1f} MB')

## Temperature sensitivity sweep

The distillation temperature `T` controls how soft the teacher's targets are.
Try a few values to see the effect on validation loss.

In [ ]:
from copy import deepcopy

sweep = []
for T in [1.0, 2.0, 4.0]:
    print(f'\n--- Temperature T={T} ---')

    new_dist = deepcopy(config.distillation).__class__(
        **{**config.distillation.__dict__, "temperature": T}
    )
    new_training = config.training.__class__(
        **{**config.training.__dict__,
           "output_dir": f"./outputs/distillation_T{int(T)}",
           "num_train_epochs": 1}    # 1 epoch to keep sweep fast
    )
    new_cfg = config.__class__(
        **{**config.__dict__, "distillation": new_dist, "training": new_training}
    )

    tr, st = build_distillation_trainer(new_cfg, train_ds, val_ds, tokenizer)
    r = tr.train()
    ev = tr.evaluate()
    sweep.append({"T": T, "train_loss": r.training_loss, "eval_loss": ev["eval_loss"]})

    del tr, st
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

import pandas as pd
df = pd.DataFrame(sweep)
print('\n\nTemperature sweep results:')
print(df.to_string(index=False))

df.to_csv("../outputs/distillation_temperature_sweep.csv", index=False)

## Summary

- Distillation loads **both** the teacher (~1B) and student (~270M).
- Loss is a mix of hard CE and soft KL against the teacher's logits.
- Temperature `T > 1` produces softer targets that improve the student.
- Typical use case: deployment — the student is **3-4× smaller** and much faster.

**Next:** `05_inference_benchmark.ipynb` to measure the distilled model's
serving speed and accuracy on the test set.